# Meu primeiro experimento com o encoder visual
**Qwen3.5 · pesos congelados · execução local**

Você controla a sequência. Leia um bloco, clique no código e pressione **Shift + Enter**. O resultado aparece logo abaixo. Não precisa copiar comandos para um terminal.

Este caderno usa as mesmas funções da tela. Você pode abrir `engine.py` na lista de arquivos do JupyterLab e examinar o que elas fazem. A execução não atualiza pesos e não usa LoRA.

**Antes de começar:** prepare sua imagem na tela e depois abra este caderno. Se preferir, o próximo bloco permite usar um padrão sintético. Evite executar a tela e o caderno ao mesmo tempo. Para liberar a memória ao terminar: **Kernel → Shut Down Kernel**.

### Minha hipótese
Edite esta célula com dois cliques. O que você espera observar? Esta nota não entra no encoder.

> Minha hipótese: ...


## 1. As ferramentas
`Path` representa caminhos; `numpy` trabalha com matrizes; `display` mostra imagens. Importar as funções da bancada ainda não carrega os pesos.


In [ ]:
from pathlib import Path
import json, inspect, sys
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import display, Code
from engine import abrir_imagem, preparar_imagem, carregar_encoder, executar_encoder, salvar_resultado

RAIZ = Path.cwd()
print("Python:", sys.version.split()[0])
print("Caderno pronto para escolher uma entrada.")


## 2. Escolha a entrada
Com `usar_demonstracao = False`, lemos a última imagem **preparada** na interface. Isso reutiliza o arquivo, sem usar resultados calculados antes. Para experimentar sem uma imagem sua, troque por `True`.

Cada vez que preparar outra imagem na tela, execute novamente este bloco e todos os blocos seguintes que dependem da imagem.


In [ ]:
usar_demonstracao = False  # Troque para True para criar um padrão geométrico.

if usar_demonstracao:
    imagem_demo = Image.new("RGB", (320, 240), "#fafafa")
    desenho = ImageDraw.Draw(imagem_demo)
    for linha in range(6):
        for coluna in range(8):
            x, y = 32 + coluna*32, 24 + linha*32
            cor = "#194fb0" if (linha+coluna) % 2 else "#f3a53a"
            desenho.rectangle((x, y, x+26, y+26), fill=cor)
    caminho = RAIZ / "sessions" / "demonstracao_caderno.png"
    caminho.parent.mkdir(exist_ok=True)
    imagem_demo.save(caminho)
    selecao = {"limit": 256, "variant": "original"}
else:
    arquivo_selecao = RAIZ / "sessions" / "latest.json"
    if not arquivo_selecao.exists():
        raise ValueError("Prepare uma imagem na tela ou mude usar_demonstracao para True.")
    selecao = json.loads(arquivo_selecao.read_text(encoding="utf-8"))
    caminho = Path(selecao["image"])

fundo = selecao.get("background", "#ffffff")
original = abrir_imagem(caminho, fundo=fundo)
print("Tamanho original em pixels:", original.size)
display(original)


## 3. Prepare os pixels
Começamos com o orçamento e a variante escolhidos na tela. Você pode substituir `selecao["limit"]` por `256`, `384` ou `512`. Esse número ao quadrado é o orçamento máximo de pixels; não é uma ordem para deixar a imagem quadrada.

Para a variante, use `"original"` ou `"cinza"`. O processador ajusta as dimensões, normaliza as cores e organiza os patches. A imagem mostrada abaixo reconstrói esses dados para podermos enxergá-los.


In [ ]:
limite = selecao["limit"]
variante = selecao["variant"]

preparacao = preparar_imagem(caminho, limite=limite, variante=variante, fundo=fundo)
display(preparacao["preparada"])
print("Tamanho preparado:", preparacao["preparada"].size)
print("Orçamento máximo:", preparacao["info"]["max_pixels"], "pixels")


## 4. Examine os números que entram
`pixel_values` é uma tabela: cada linha reúne **1.536 números** de um patch: 3 canais × 2 cópias temporais × 16 × 16 pixels. A repetição temporal permite que a arquitetura processe uma imagem estática; não cria movimento.

`image_grid_thw` informa a grade temporal, altura e largura em patches. Para 256 × 256 pixels, por exemplo, teremos uma grade 1 × 16 × 16 e 256 linhas. A forma real abaixo depende da sua imagem.


In [ ]:
entradas = preparacao["inputs"]
print("pixel_values:", tuple(entradas["pixel_values"].shape))
print("image_grid_thw:", entradas["image_grid_thw"].tolist())
print("Tipo dos números:", entradas["pixel_values"].dtype)
print("Intervalo:", preparacao["info"]["intervalo"])
print("Primeiros 12 números do primeiro patch:")
display(entradas["pixel_values"][0, :12])


### Uma normalização que dá para calcular à mão
Neste checkpoint, a média e o desvio são 0,5 por canal. Assim, `(cor / 255 − 0,5) / 0,5` transforma 0 em −1 e 255 em +1. A tabela completa não é uma lista de rótulos: são números de entrada.


In [ ]:
cor_8_bits = np.array([0, 128, 255], dtype=float)
normalizada = (cor_8_bits / 255 - 0.5) / 0.5
print("Exemplo RGB:", cor_8_bits)
print("Normalizado:", normalizada)
print("Primeiro pixel real, canais R/G/B:", preparacao["normalizada"][:, 0, 0].tolist())


### Abrir a função da preparação
O código abaixo é o da função que acabou de ser executada. A organização dos patches e sua reconstrução visual ficam visíveis aqui; as operações internas do processador pertencem à biblioteca Transformers.


In [ ]:
display(Code(inspect.getsource(preparar_imagem), language="python"))


## 5. Carregue só a torre visual
Agora alocamos memória para o encoder e carregamos os pesos com `strict=True`: nomes ou formas incompatíveis fazem a operação falhar. O cálculo usa CPU e FP32; o arquivo de pesos BF16 permanece intacto.

Esta etapa pode levar alguns segundos. Ela ainda não passa a imagem pela rede. `eval()` desativa comportamentos de treinamento; os parâmetros também ficam com gradientes desativados.


In [ ]:
modelo = carregar_encoder(lambda texto, bloco: print(texto))
print("Blocos visuais:", len(modelo.blocks))
print("Parâmetros:", f"{sum(p.numel() for p in modelo.parameters()):,}")
print("Tipo de cálculo:", modelo.dtype)
print("Algum parâmetro com gradiente?", any(p.requires_grad for p in modelo.parameters()))


## 6. Execute a imagem
O progresso abaixo é emitido quando cada bloco termina. Não são porcentagens simuladas. A função chama a torre visual com `hidden_states=pixel_values` e `grid_thw=image_grid_thw` dentro de `torch.inference_mode()`.

Ela recolhe duas saídas: `last_hidden_state` antes do merger e `pooler_output` depois dele. **Antes e depois aqui são pontos da mesma passagem pela rede; não houve treinamento entre eles.**


In [ ]:
resultado = executar_encoder(preparacao, lambda texto, bloco: print(texto))
antes = resultado["antes"]
depois = resultado["depois"]
print("Antes do merger:", antes.shape)
print("Depois do merger:", depois.shape)
print("Inferência em segundos:", resultado["info"]["tempo_inferencia_s"])


### O que são essas formas?
Antes do merger: uma linha por posição de patch e **1.152 dimensões**. Depois: grupos espaciais de quatro posições são reunidos e projetados em **4.096 dimensões**. A atenção já misturou informações entre posições; uma linha não representa só os pixels de um quadradinho isolado.

Vamos mostrar apenas 4 posições × 8 dimensões para a tabela caber. As matrizes completas continuam nas variáveis `antes` e `depois`.


In [ ]:
np.set_printoptions(precision=4, suppress=True)
print("Antes do merger — primeiras posições e dimensões:")
display(antes[:4, :8])
print("Depois do merger — primeiras posições e dimensões:")
display(depois[:4, :8])
print("Todos os valores são finitos?", np.isfinite(antes).all() and np.isfinite(depois).all())


## 7. Nossa escolha: um vetor por imagem
A rede devolveu várias posições. Para este primeiro ensaio, **nós** escolhemos calcular a média das posições depois do merger. `axis=0` soma pelas linhas, preservando as 4.096 coordenadas.

Depois dividimos pelo comprimento do vetor (norma L2). Isso prepara o vetor para comparações por cosseno. Não há nomes de classes associados às coordenadas. Outra camada ou outra forma de resumir as posições pode produzir uma geometria diferente.


In [ ]:
vetor_medio = depois.mean(axis=0)
comprimento = np.linalg.norm(vetor_medio)
vetor_unitario = vetor_medio / comprimento

print("Forma:", vetor_unitario.shape)
print("Comprimento depois da normalização:", np.linalg.norm(vetor_unitario))
print("Primeiras 12 coordenadas:", vetor_unitario[:12])
print("Conferência da receita padrão (média + L2):", np.allclose(vetor_unitario, resultado["vetor_unitario"]))


### Ver o código da execução completa
Aqui você pode conferir o modo de inferência, os pontos de coleta e o resumo por média. O arquivo `engine.py` pode ser aberto e editado no JupyterLab. Depois de editar uma função importada, reinicie o kernel e execute o caderno novamente para evitar usar a versão antiga em memória.


In [ ]:
display(Code(inspect.getsource(executar_encoder), language="python"))


## 8. Experimento opcional: cor e tons de cinza
Este bloco só executa uma segunda imagem se você mudar `comparar_agora` para `True`. Escolha uma variante diferente da usada anteriormente. Mantemos o mesmo arquivo e orçamento de pixels.

O produto escalar de dois vetores normalizados é o cosseno entre eles. Valores próximos de 1 indicam direções próximas nesse espaço, segundo **esta** escolha de representação. `0,58` não significa “58% parecido”. Uma mudança mostra sensibilidade à transformação; não prova compreensão nem erro do modelo.


In [ ]:
comparar_agora = False
outra_variante = "cinza" if variante == "original" else "original"

if comparar_agora:
    outra_preparacao = preparar_imagem(caminho, limite=limite, variante=outra_variante, fundo=fundo)
    display(outra_preparacao["preparada"])
    outro_resultado = executar_encoder(outra_preparacao)
    cosseno = np.dot(vetor_unitario, outro_resultado["vetor_unitario"])
    print(f"Cosseno entre {variante} e {outra_variante}: {cosseno:.6f}")
else:
    print("Comparação não executada. Altere comparar_agora se quiser testar.")


## Experimente outra receita com os mesmos tokens

Esta é uma escolha metodológica da bancada. O merger e os pesos ficam intactos.
O bloco abaixo reaproveita `resultado`, sem executar a rede novamente. Escolha:

- `pooling`: `mean` (média), `max` (máximo por coordenada) ou `median`;
- `normalization`: `l2`, `l1`, `linf` ou `none`;
- `metric`: `cosine`, `euclidean` ou `dot`.

L1/L2/L∞ mudam a escala, preservando a direção de vetores não nulos; só trocar entre elas não muda matematicamente o cosseno. A medida fica registrada e será usada ao comparar vetores compatíveis. Imagem individual não tem vizinhos sem um conjunto.

O campo `analysis` contém a nova receita. Os campos históricos `vetor_medio` e `vetor_unitario` continuam descrevendo média + L2, para compatibilidade. O bloco de exportação a seguir guarda ambos explicitamente.


In [ ]:
from engine import reanalisar
from analysis import summary

receita = {"version": 1, "pooling": "mean", "normalization": "l2", "metric": "cosine"}
resultado = reanalisar(resultado, receita)
vetor_escolhido = resultado["analysis"]["after"]  # "before" para antes do merger
print(summary(resultado))
print("Primeiras 12 coordenadas da receita escolhida:", vetor_escolhido[:12])


## 9. Guarde o experimento
O arquivo `.npz` guarda as matrizes completas e os vetores. O registro `.json` guarda a preparação e as condições de execução. Uma nova pasta é criada a cada execução deste bloco.


In [ ]:
from datetime import datetime

pasta_resultado = RAIZ / "sessions" / ("caderno_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
pasta_resultado.mkdir(parents=True)
salvar_resultado(resultado, pasta_resultado / "representacoes.npz")
from analysis import summary
registro = {"preparacao": preparacao["info"], "execucao": resultado["info"], "analysis": summary(resultado)}
(pasta_resultado / "registro.json").write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")
print("Salvo em:", pasta_resultado.name)


## Minhas observações
**O que eu esperava:** ...

**O que apareceu:** ...

**Qual condição mudei:** ...

**O que ainda não consigo concluir:** ...

## O que este caderno ainda não mostra
Não coletamos todas as ativações intermediárias nem a atenção. Não atribuímos significado isolado às coordenadas. Não comparamos com um conjunto de referência nem criamos agrupamentos. Esses são próximos experimentos; o objetivo agora é tornar a primeira extração observável e reproduzível.

## Referências de implementação
- [Checkpoint oficial Qwen3.5-9B](https://huggingface.co/Qwen/Qwen3.5-9B)
- [Código da arquitetura Qwen3.5 em Transformers](https://github.com/huggingface/transformers/blob/main/src/transformers/models/qwen3_5/modeling_qwen3_5.py)

O registro de cada execução informa a versão **local** efetivamente usada. A implementação dos links pode mudar ao longo do tempo.


## Transparência e lote na interface
A aba **Uma imagem** continua independente da aba **Lote de imagens**. As condições do lote não alteram as escolhidas para uma imagem.

O encoder recebe **RGB**, sem canal alfa. Quando existe transparência, compomos a imagem sobre o fundo escolhido. O original RGBA e a máscara de opacidade ficam preservados separadamente. Branco opaco tem alfa 255; uma região totalmente transparente tem alfa 0. A máscara mostra isso, sem inferir quais pixels são conteúdo ou fundo.

O campo `fundo` acima recebe a escolha da última preparação na interface. Você também pode escrever uma cor como `"#808080"` e executar novamente as etapas dependentes. A imagem original com transparência está em `preparacao["original_rgba"]`; a máscara está em `preparacao["alpha"]`. O valor `limite = 0` usa os limites do checkpoint, como o script do terminal.

**As barras:** as 48 primeiras coordenadas são uma amostra por ordem, não as mais importantes. Positivo e negativo são lados de zero num eixo aprendido, não acerto/erro ou confiança. A média e as transformações da rede podem ter ambos os sinais; a divisão pela norma positiva não troca esses sinais.
